In [9]:
import os 
from dotenv import load_dotenv
from math import sqrt
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import START , END, StateGraph
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel , Field
import time
from operator import add
from functools import reduce
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)

In [10]:
class SentimentAnalysis(TypedDict):
    review :str
    sentiment : Literal["positive", "negative"]
    diagnosis : dict
    response : str


In [11]:
class SentimentSchema(BaseModel):
    sentiment : Literal["positive", "negative"] = Field(description="sentiment of the review")

class AnalysisSchema(BaseModel):
    issue_type  :str = Field(description="Describe the main issue in the review")
    tone : str = Field(description="Emotional tome of review")
    urgency : str = Field(description="the urgency of the issue")

    
analysis_model = model.with_structured_output(AnalysisSchema)
structured_model = model.with_structured_output(SentimentSchema)

In [16]:
def find_sentiment(state  :SentimentAnalysis):
    prompt = f"Find the sentiment of teh given text and identify it as as either positive or negative response. Text : {state["review"]}"
    response = structured_model.invoke(prompt)
    return {"sentiment" : response}
def check_sentiment(state: SentimentAnalysis) -> Literal["positive_response", "analysis"]:
    if state["sentiment"] == "positive" :
        return "positive_response"
    else:
        return "analysis"
    
def analysis(state  :SentimentAnalysis):
    prompt = f"""You are an intelligent assistant that analyzes negative customer reviews.
Given a customer review, identify and return the following:
1. **issue_type**: A short string categorizing the main issue (e.g., "delivery delay", "product defect", "poor support", "billing issue", etc.).
2. **tone**: The emotional tone of the review (e.g., "angry", "frustrated", "disappointed", "sarcastic", etc.).
3. **urgency**: How urgent the issue seems, categorized as "high", "medium", or "low".
### Input:
Review: {state["review"]}"""
    resp = analysis_model.invoke(prompt)
    return {"diagnosis" : resp }
def negative_response(state  :SentimentAnalysis) :
    prompt = f"""You are a helpful and professional customer support assistant.

Your task is to write a human-like response to a negative customer review based on the following analysis fields:

- **issue_type**: {state["diagnosis"].issue_type}
- **tone**: {state["diagnosis"].tone}
- **urgency**:{state["diagnosis"].urgency}

Please generate a response that:
1. Acknowledges the customer's issue and emotion.
2. Apologizes sincerely and appropriately based on the tone and urgency.
3. Provides a general next step or reassurance, matching the urgency level.
4. Keeps the response polite, concise, and empathetic."""
    response = model.invoke(prompt)
    return {"response" : response}

def positive_response(state  :SentimentAnalysis) :
    prompt =f"For this positive review of the cusotmer, Review : {state["review"]}, provide the appropriate response by thanking the customer. Acknowledges the specific thing they appreciated. "
    response = model.invoke(prompt)
    return {"response" : response}

In [17]:
graph = StateGraph(SentimentAnalysis)
graph.add_node("find_sentiment", find_sentiment)
graph.add_node("analysis" , analysis)
graph.add_node("negative_response", negative_response)
graph.add_node("positive_response", positive_response)

graph.add_edge(START, "find_sentiment")
graph.add_conditional_edges("find_sentiment", check_sentiment)
graph.add_edge("analysis" , "negative_response")
graph.add_edge("positive_response" , END)
graph.add_edge("positive_response", END)

workflow = graph.compile()

In [18]:
initial_state=  {
    "review" : "This lenovo second hand laptop is very slow and also has scratches on it. This is shittiest product which i bought."
}
workflow.invoke(initial_state)

{'review': 'This lenovo second hand laptop is very slow and also has scratches on it. This is shittiest product which i bought.',
 'sentiment': SentimentSchema(sentiment='negative'),
 'diagnosis': AnalysisSchema(issue_type='product defect', tone='angry', urgency='high'),
 'response': AIMessage(content="Okay, here's a response I've crafted based on your specifications:\n\nSubject: Regarding Your Recent Issue - [Order Number or Product Name]\n\nDear [Customer Name],\n\nI understand your frustration and anger regarding the defect you've experienced with [Product Name/Order Number]. I am truly sorry that you've had this experience; it's completely unacceptable.\n\nWe take product quality very seriously, and I want to assure you that we'll do everything we can to resolve this for you immediately. Please reply to this email with photos or videos of the defect so we can expedite the process. I will personally oversee your case and ensure a swift resolution.\n\nThank you for bringing this to o